In [ ]:
#Install requirements
%pip install -r "../Requirements.txt"

In [ ]:
#Import libraries
import pandas as pd 
import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
from pathlib import Path
import scanpy as sc
from scipy.sparse import issparse
import sys
import re
from matplotlib.patches import Patch


In [ ]:
#Add the src to the path
sys.path.append(os.path.abspath(os.path.join('..')))


In [ ]:
#Define paths for original data and output directories
data_dir = Path('../data/GSE107451_DGRP-551_w1118_WholeBrain_57k_0d_1d_3d_6d_9d_15d_30d_50d_10X_DGEM_MEX.mtx.tsv') #folder with the matrix, genes.tsv, barcodes.tsv
annot_path = Path('../data/GSE107451_DGRP-551_w1118_WholeBrain_57k_0d_1d_3d_6d_9d_15d_30d_50d_10X_DGEM_MEX.mtx.tsv/annotation.tsv') #annotation file
metadata = Path('../data/GSE107451_DGRP-551_w1118_WholeBrain_57k_Metadata.tsv/GSE107451_DGRP-551_w1118_WholeBrain_57k_Metadata.tsv') #metadata

output_dir = Path('../data/preprocessing')
output_dir.mkdir(exist_ok=True)

output_dir_figs = Path('../figures/preprocessing')
output_dir_figs.mkdir(exist_ok=True)

In [ ]:
#Load the Matrix
adata = sc.read_10x_mtx(data_dir, var_names="gene_symbols", cache=False)

print(f"The matrix loaded successfully. It consists of {adata.n_obs} cells x {adata.n_vars} genes")

In [ ]:
#Load metadata and merge some columns
meta = pd.read_csv(metadata, sep = "\t") #cell annotation, QC metrics, Genotype, sex, age
barcode_col = "new_barcode"
meta = meta.set_index(barcode_col) #set the index of the metadata file to be the new_barcode column

#columns from the metadata file that we need
meta_cols_to_keep = [
    "annotation", # cell type label
    "Age", #timepoints in days
    "Replicate", #true biological replicates
    "Genotype", #line
    "nGene",  # number of genes detected
    "nUMI",  # total UMI counts
    "percent.mito", # mitochondrial gene percentage
    "sex"
]

#check if the columns exist
meta_cols_to_keep = [c for c in meta_cols_to_keep if c in meta.columns]
meta = meta[meta_cols_to_keep]

#Merge the information from the metadata into the adata objects
adata.obs = adata.obs.join(meta, how="left") #.join() joins on the index by default

In [ ]:
print(f"adata.obs columns {list(adata.obs.columns)}")
print(f"Unique lines: {adata.obs['Genotype'].unique()}")
print(f"Cells missing cell type label:  {adata.obs['annotation'].isna().sum():,}")
print(f"Unique time points: {sorted(adata.obs['Age'].unique())}")
print(f"Unique cell types: {adata.obs['annotation'].nunique()}")

In [ ]:
#Selected timepoints.
timepoints = [0, 1, 3, 6, 9,15,30,50] #very young, young, mid-young, mid-aging, aging. 50 days is missing in one genotype

age_map = {
    0: "eclosion",
    1: "1_very_young",
    3: "2_very_young",
    6: "1_young",
    9: "2_young",
    15: "mid_young",
    30: "mid_aged",
    50: "aged"
}

#Filter for the selected timepoints
adata = adata[adata.obs["Age"].isin(timepoints)].copy()

adata.obs["age_map"] = adata.obs["Age"].map(age_map)

print(adata.obs.groupby(["Genotype", "Age"]).size().unstack(fill_value=0).to_string())

In [ ]:
age_map_order = ["eclosion", "1_very_young", "2_very_young", "1_young", "2_young", "mid_young", "mid_aged", "aged"]
age_map_colors = ["#2C3E7A",  "#3A9BD5", "#5CB85C",  "#A8D44A",  "#F0AD4E", "#E8733A",  "#C0392B", "#7B1E1E"]

In [ ]:
#Remove cell clusters with unknown/non-biologically meaningfull label (numeric labels)
from src.preprocessing_functions import is_named

named_mask = adata.obs["annotation"].apply(is_named)
removed_clusters = sorted(
    adata.obs.loc[~named_mask, "annotation"].dropna().unique(), key=lambda x: int(x) if str(x).strip().isdigit() else 0)

print(f"Removed clusters: {removed_clusters}")

adata = adata[named_mask].copy()
print(f"{adata.obs['annotation'].nunique()} named cell types")

In [ ]:
#Detect clusters that are too small for reliable ML (<100 cells)
cluster_sizes = adata.obs["annotation"].value_counts()
small_clusters = cluster_sizes[cluster_sizes < 100].index.tolist()
print(f"Found {len(small_clusters)} clusters with <100 cells")

#Store usable clusters
usable_clusters = cluster_sizes.index.tolist()
pd.Series(usable_clusters).to_csv(
    output_dir / "usable_clusters.csv", index=False, header=["annotation"]
)

In [ ]:
#Normalize expression with the same way in both pipelines
adata.raw = adata
adata.layers["raw_counts"] = adata.X.copy()

In [ ]:
#Visualizations before preprocessing steps

n_sample = min(3000, adata.n_obs)
idx_sample = np.random.choice(adata.n_obs, n_sample, replace=False) #select some of the cells (not all 57,000) just for visualization
X_sample = adata.X[idx_sample]

if issparse(X_sample):
    X_sample = X_sample.toarray()

raw_nonzero = X_sample[X_sample >0].flatten() #visualize only the nonzero expression genes at this step

fig, axes = plt.subplots(1,3, figsize=(15,4))

#Distribution of raw counts - we expect right-skewed distribution
axes[0].hist(raw_nonzero, bins=100, color="#2E75B6", edgecolor="none", log=True)
axes[0].set_title("Raw UMI counts")
axes[0].set_xlabel("UMI count")
axes[0].set_ylabel("Frequency (log)")

#Total UMIs per cell
axes[1].hist(adata.obs["nUMI"].dropna(), bins=80, color='#E8733A', edgecolor="none")
axes[1].set_title("Total UMIs per cell")
axes[1].set_xlabel("Total UMI count")
axes[1].set_ylabel("Number of cells")

#Fraction of zeros per cell
zero_frac = (X_sample ==0).mean(axis=1)
axes[2].hist(zero_frac, bins=60, color="#5CB85C", edgecolor="none")
axes[2].axvline(zero_frac.mean(), color="red", linestyle="--", label=f"Mean: {zero_frac.mean(): .1%}")
axes[2].set_title("Fraction of zeros per cell")
axes[2].set_xlabel("Fraction of genes = 0")
axes[2].set_ylabel("Number of cells")
axes[2].legend()


plt.suptitle("Expression matrix before normalization", fontsize=13)
plt.tight_layout()
plt.savefig(output_dir_figs / "before_normalization.png", dpi=300)
plt.show()

In [ ]:
#Scale every cell to 10,000 total counts
sc.pp.normalize_total(adata, target_sum=1e4)
#Log transform - makes the distribution more normal
sc.pp.log1p(adata)
print(f" normalize_total(10k) + log1p applied")

adata.layers["normalised"] = adata.X.copy()

In [ ]:
#Visualizations  after normalization

X_norm_sample = adata.X[idx_sample]

if issparse(X_norm_sample):
    X_norm_sample = X_norm_sample.toarray()

norm_nonzero = X_norm_sample[X_norm_sample >0].flatten() #visualize only the nonzero expression genes 

fig, axes = plt.subplots(1,3, figsize=(16,4))

#Distribution of raw counts before
axes[0].hist(raw_nonzero, bins=100, color="#AAAAAA", edgecolor="none", log=True)
axes[0].set_title("Raw UMI counts")
axes[0].set_xlabel("UMI count")
axes[0].set_ylabel("Frequency (log)")

#after normalization
axes[1].hist(norm_nonzero, bins=100, color="#2E75B6", edgecolor="none")
axes[1].set_title("After normalization + log1p")
axes[1].set_xlabel("log1p expression value")
axes[1].set_ylabel("Frequency")
axes[1].set_xlim(0, 12) 

#Gene mean vs variance
gene_mean = np.array(adata.X.mean(axis=0)).flatten()
if issparse(adata.X):
    gene_var = np.array(adata.X.power(2).mean(axis=0)).flatten() - gene_mean**2
else:
    gene_var = np.var(adata.X, axis=0)
 
n_genes = len(gene_mean)
plot_idx = np.random.choice(n_genes, min(5000, n_genes), replace=False)
axes[2].scatter(gene_mean[plot_idx], gene_var[plot_idx],
                alpha=0.2, s=2, color="#2E75B6")
axes[2].set_xscale("log")
axes[2].set_yscale("log")
axes[2].set_title("Mean vs variance per gene (after normalization)")
axes[2].set_xlabel("Mean expression (log)")
axes[2].set_ylabel("Variance (log)")


plt.suptitle("Expression matrix after normalization", fontsize=13)
plt.tight_layout()
plt.savefig(output_dir_figs / "after_normalization.png", dpi=300)
plt.show()

In [ ]:
#Select high variance genes (HVG) for visualization only - in the ML pipeline we will use all the features until feature selection
sc.pp.highly_variable_genes(adata, n_top_genes=3000, flavor="seurat_v3")
n_hvg = adata.var["highly_variable"].sum()

#Perform PCA 
sc.pp.pca(adata, n_comps=50, use_highly_variable=True)
sc.pp.neighbors(adata, n_neighbors=15, n_pcs=40)
sc.tl.umap(adata)

#Store UMAP
adata.obsm["X_umap_before"] = adata.obsm["X_umap"].copy()

In [ ]:
#Visualize cells per cluster
fig,ax = plt.subplots(figsize=(16,5))
color_bar = ["#2E75B6" if n >=100 else "#C0392B" for n in cluster_sizes.values]
ax.bar(range(len(cluster_sizes)), cluster_sizes.values, color = color_bar, edgecolor="none", width=0.8)
ax.set_xticks(range(len(cluster_sizes)))
ax.set_xticklabels(cluster_sizes.index, rotation=90, fontsize=7)
ax.axhline(100, color="red", linestyle="--", linewidth=0.8)
ax.set_title("Number of cells per annotated cluster", fontsize=13)
ax.set_xlabel("Cell type")
ax.set_ylabel("Number of cells")


ax.legend(handles=[
    Patch(facecolor="#2E75B6", label="≥100 cells"),
    Patch(facecolor="#C0392B", label="<100 cells"),
    plt.Line2D([0],[0], color="red", ls="--", label="100-cell threshold")
])
plt.tight_layout()
plt.savefig(output_dir_figs / "cells_per_cluster.png", dpi=300)
plt.show()


In [ ]:
#Visualize total dataset composition
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col, title, color in zip(axes,
    ["Genotype", "sex",     "Age"],
    ["Genotype",   "Sex",     "Age (days)"],
    ["#2E75B6",  "#E8733A", "#5CB85C"]):
    counts = adata.obs[col].value_counts().sort_index()
    ax.bar(counts.index.astype(str), counts.values,
           color=color, edgecolor="none")
    ax.set_title(title, fontsize=12)
    ax.set_ylabel("Number of cells")
    for t in ax.get_xticklabels():
        t.set_rotation(30); t.set_ha("right")
plt.suptitle("Dataset composition overview", fontsize=13)
plt.tight_layout()
plt.savefig(output_dir_figs /"dataset_composition.png", dpi=300)
plt.show()


In [ ]:
#Visualize Age × Genotype (heatmap)
ct = pd.crosstab(adata.obs["Genotype"], adata.obs["Age"])
fig, ax = plt.subplots(figsize=(10, 3))
sns.heatmap(ct, annot=True, fmt=",", cmap="Blues", ax=ax,
            linewidths=0.3, cbar_kws={"label": "Number of cells"})
ax.set_title("Cells per age × Genotype", fontsize=12)
ax.set_xlabel("Age (days)")
ax.set_ylabel("Genotype")
plt.tight_layout()
plt.savefig(output_dir_figs /"age_by_genotype.png", dpi=300)
plt.show()


In [ ]:
#Visualize Sex × Genotype
ct2 = pd.crosstab(adata.obs["Genotype"], adata.obs["sex"])
fig, ax = plt.subplots(figsize=(6, 3))
ct2.plot(kind="bar", ax=ax, color=["#E8733A","#2E75B6"], edgecolor="none")
ax.set_title("Cells per sex × Genotype", fontsize=12)
ax.set_xlabel("Genotype")
ax.set_ylabel("Number of cells")
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
ax.legend(title="Sex")
plt.tight_layout()
plt.savefig(output_dir_figs/"sex_by_genotype.png", dpi=300)
plt.show()


In [ ]:
#Visualize the target variable: Age distribution 

fig, ax = plt.subplots(figsize=(10, 4))

bin_counts = adata.obs["age_map"].value_counts().reindex(age_map_order, fill_value=0)
wrapped_labels = [label.replace("_", "\n") for label in age_map_order]

ax.bar(
    range(len(age_map_order)),
    bin_counts.values,
    color=age_map_colors,
    edgecolor="none"
)
ax.set_xticks(range(len(age_map_order)))
ax.set_xticklabels(wrapped_labels, fontsize=9, ha="center")
ax.set_title("Age bin distribution")
ax.set_ylabel("Number of cells")


for i, v in enumerate(bin_counts.values):
    ax.text(i, v + 50, f"{v:,}", ha="center", fontsize=9)

plt.tight_layout()
plt.savefig(output_dir_figs / "age_bin_distribution.png", dpi=300)
plt.show()


In [ ]:
#Visualize Age distribution per cluster (boxplot)

subset_obs = adata.obs[adata.obs["annotation"].isin(usable_clusters)]
median_order = (subset_obs.groupby("annotation")["Age"]
                .median().sort_values().index.tolist())
 
fig, ax = plt.subplots(figsize=(18, 5))
bp_data = [subset_obs[subset_obs["annotation"] == ct]["Age"].values
           for ct in median_order]
bp = ax.boxplot(bp_data, patch_artist=True,
                medianprops=dict(color="white", linewidth=1.5))
for patch in bp["boxes"]:
    patch.set_facecolor("#2E75B6")
    patch.set_alpha(0.7)
ax.set_xticks(range(1, len(median_order)+1))
ax.set_xticklabels(median_order, rotation=90, fontsize=6)
ax.set_title("Age distribution per cell type cluster",fontsize=12)
ax.set_xlabel("Cell type")
ax.set_ylabel("Age (days)")
ax.set_yticks(sorted(adata.obs["Age"].dropna().unique()))
plt.tight_layout()
plt.savefig(output_dir_figs/"age_per_cluster_boxplot.png", dpi=300)
plt.show()


In [ ]:
#Visualize Age bin × cluster (heatmap) - class imbalance for each cluster

pivot = (adata.obs[adata.obs["annotation"].isin(usable_clusters)]
         .groupby(["annotation","age_map"]).size()
         .unstack(fill_value=0)
         .reindex(columns=age_map_order, fill_value=0))
 
fig, ax = plt.subplots(figsize=(8, max(6, len(usable_clusters) * 0.22)))
sns.heatmap(pivot, annot=True, fmt="d", cmap="YlOrRd",
            ax=ax, linewidths=0.2,
            cbar_kws={"label": "Number of cells"},
            xticklabels=age_map_order)
ax.set_title("Cells per age bin × cell type", fontsize=12)
ax.set_xlabel("Age bin")
ax.set_ylabel("Cell type")
ax.tick_params(axis="y", labelsize=7)
plt.tight_layout()
plt.savefig(output_dir_figs/"agebin_cluster_heatmap.png",
            dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
#Visualize imbalance ratio per cluster
imbalance = pivot.apply(
    lambda r: r.max() / r[r > 0].min() if (r > 0).sum() > 1 else np.nan,
    axis=1
).dropna().sort_values(ascending=False)
 
fig, ax = plt.subplots(figsize=(16, 4))
bar_colors = ["#C0392B" if v > 10 else "#E8733A" if v > 5 else "#2E75B6"
              for v in imbalance.values]
ax.bar(range(len(imbalance)), imbalance.values,
       color=bar_colors, edgecolor="none")
ax.axhline(5,  color="#E8733A", linestyle="--", linewidth=0.8, label="5× ratio")
ax.axhline(10, color="#C0392B", linestyle="--", linewidth=0.8, label="10× ratio")
ax.set_xticks(range(len(imbalance)))
ax.set_xticklabels(imbalance.index, rotation=90, fontsize=7)
ax.set_title("Class imbalance ratio per cluster", fontsize=12)
ax.set_ylabel("Imbalance ratio")
ax.legend()
plt.tight_layout()
plt.savefig(output_dir_figs/"imbalance_ratio.png", dpi=300)
plt.show()

In [ ]:
#Visualize gene expression 
n_sample = min(3000, adata.n_obs) #sample of cells - not all the cells
idx_s= np.random.choice(adata.n_obs, n_sample, replace=False)
X_s= adata.X[idx_s]
if issparse(X_s):
    X_s = X_s.toarray()
 
#Sparsity
zero_frac = (X_s == 0).mean(axis=1)
fig, ax   = plt.subplots(figsize=(8, 4))
ax.hist(zero_frac, bins=60, color="#2E75B6", edgecolor="none")
ax.axvline(zero_frac.mean(), color="red", linestyle="--",
           label=f"Mean: {zero_frac.mean():.1%} zeros")
ax.set_title("Sparsity: fraction of zero-expressed genes per cell",fontsize=12) #most genes non informative
ax.set_xlabel("Fraction of genes = 0")
ax.set_ylabel("Number of cells")
ax.xaxis.set_major_formatter(ticker.PercentFormatter(xmax=1))
ax.legend()
plt.tight_layout()
plt.savefig(output_dir_figs /"sparsity.png", dpi=300)
plt.show()


In [ ]:
 #Visualize Mean vs variance
gene_mean = np.array(adata.X.mean(axis=0)).flatten()
if issparse(adata.X):
    gene_var = np.array(adata.X.power(2).mean(axis=0)).flatten() - gene_mean**2
else:
    gene_var = np.var(adata.X, axis=0)
 
n_genes  = len(gene_mean)
plot_idx = np.random.choice(n_genes, min(5000, n_genes), replace=False)
hvg_flag = adata.var.get("highly_variable", pd.Series(False, index=adata.var_names))
hvg_arr  = hvg_flag.values[plot_idx]
 
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(gene_mean[plot_idx][~hvg_arr], gene_var[plot_idx][~hvg_arr],
           alpha=0.2, s=2, color="#AAAAAA", label="Other genes")
ax.scatter(gene_mean[plot_idx][hvg_arr], gene_var[plot_idx][hvg_arr],
           alpha=0.5, s=4, color="#2E75B6", label="Highly variable genes")
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_title("Mean expression vs variance per gene", fontsize=12)
ax.set_xlabel("Mean expression (log)")
ax.set_ylabel("Variance (log)")
ax.legend()
plt.tight_layout()
plt.savefig(output_dir_figs/"mean_variance.png", dpi=300)
plt.show()
